In [6]:

from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import openai
import json
from threading import Lock
import pandas as pd
import re
import time
import nltk
import spacy
import numpy as np
from collections import Counter
from threading import local

# Load English tokenizer, POS tagger, parser, NER, and word vectors
nlp = spacy.load("en_core_web_sm")

# Define a word list using nltk corpus
english_vocab = set(w.lower() for w in nltk.corpus.words.words())

open_ai_keys = [os.getenv(f"OPENAI_KEY_{i}") for i in range(1, 26)]

2023-06-20 02:46:18.006772: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
original_transcript_path = "../runs/2008-09-26-debate-preclean.txt"

In [ ]:
def process_transcript(file_path, speakers_set):
    with open(file_path, 'r') as file:
        lines = file.readlines()

    # Initialize an empty list to store the speakers and their speeches
    processed_transcript = []

    # Initialize an empty string to store the current speaker
    current_speaker = ''

    for line in lines:
        # Remove leading and trailing whitespaces
        line = line.strip()

        if not line:
            continue

        # Split the line into speaker and speech, the maxsplit parameter is set to 1
        split_line = line.split(':', 1)

        # Check if the line could be split into speaker and speech
        if len(split_line) == 2:
            # Potential speaker
            potential_speaker = split_line[0].strip().upper()
            
            # If the potential speaker is in our known speakers set
            if potential_speaker in speakers_set:
                # Update the current speaker and append the speech to the transcript
                current_speaker, speech = split_line
                current_speaker = current_speaker.strip().upper()
                speech = speech.strip()
                
                processed_transcript.append((current_speaker, speech))
            else:
                # If the line could not be split into a valid speaker and speech, append it to the last speaker's last speech
                processed_transcript[-1] = (processed_transcript[-1][0], processed_transcript[-1][1] + ' ' + line)
        else:
            # If the line could not be split, append it to the last speaker's last speech
            processed_transcript[-1] = (processed_transcript[-1][0], processed_transcript[-1][1] + ' ' + line)

    # Create a dictionary to count speaker turns
    speaker_turns_dict = {speaker: sum(1 for t in processed_transcript if t[0] == speaker) for speaker in speakers_set}

    return processed_transcript, speaker_turns_dict

In [ ]:
def write_transcript_to_file(transcript, output_path):
    with open(output_path, 'w') as file:
        for speaker_name, speaker_content in transcript:
            # Each line starts with the speaker name followed by their speech
            file.write(f'{speaker_name}: {speaker_content}\n')


In [ ]:
known_speakers = {'LEHRER', 'OBAMA', 'MCCAIN'}

In [ ]:
processed_transcript, speaker_turns_dict = process_transcript(original_transcript_path, known_speakers)

output_file_name = '2008-9-29-clean-final.txt'

write_transcript_to_file(processed_transcript, output_file_name)

In [ ]:
def transcript_to_dataframe(processed_transcript):
    df = pd.DataFrame(processed_transcript, columns=['speaker', 'content'])
    return df

transcript_df = transcript_to_dataframe(processed_transcript)

In [ ]:
def configure_transcript_dataframe(df):
    df['turn_number'] = df.index + 1
    df['cumulative_speaker_turn_number'] = df.groupby('speaker').cumcount() + 1

    default_values = {
    'relevancy': 0,
    'turn_topic': 'empty',
    'word_count_points': 0,
    'sentence_count_points': 0,
    'named_entity_count_points': 0,
    'named_entities': 'empty',
    'bonus_word_points': 0,
    'bonus_word_words': 'empty',
    'base_score': 0.0,
    'E': 0.0, # change
    'P': 0.0, # change
    'L': 0.0, # change
    'E_expl': 'empty',
    'P_expl': 'empty',
    'L_expl': 'empty',
    'epl_multiplier': 0.0,
    'total_turn_score': 0.0,
    'cumulative_speaker_base_score': 0.0,
    'cumulative_speaker_total_score': 0.0,
    }

    for col, default_value in default_values.items():
        df[col] = default_value

    return df

scoreboard_df = configure_transcript_dataframe(transcript_df)





scoreboard_df

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

def calculate_relevancy(current_turn, previous_turn):
    # Combine the current turn with the previous turn
    texts = [previous_turn, current_turn]

    # Create the vectorizer
    vectorizer = TfidfVectorizer()

    # Transform the texts into TF-IDF vectors
    tfidf_matrix = vectorizer.fit_transform(texts)

    # Calculate the cosine similarity of the current turn to the previous turn
    similarity_matrix = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])

    # Return the similarity as the relevancy
    return similarity_matrix[0][0]


In [ ]:
# Loop through the dataframe from the second row to the end
for i in range(1, len(scoreboard_df)):
    # Calculate the relevancy of the current row compared to the previous row
    relevancy = calculate_relevancy(scoreboard_df.loc[i, 'content'], scoreboard_df.loc[i-1, 'content'])
    
    # Store the relevancy in the dataframe
    scoreboard_df.loc[i, 'relevancy'] = relevancy

In [ ]:
scoreboard_df

In [ ]:
def generate_speaker_dfs(scoreboard_df):
    # Get unique speaker names
    speakers = scoreboard_df['speaker'].unique()

    if len(speakers) != 3:
        raise ValueError("The transcript should contain exactly 3 unique speakers.")

    # Initialize a list to hold dataframes
    speaker_dfs = []

    # Iterate over speakers
    for speaker in speakers:
        # Create a mask for the current speaker
        speaker_mask = scoreboard_df['speaker'] == speaker
        
        # Extract turns for the current speaker
        speaker_df = scoreboard_df.loc[speaker_mask, ['turn_number', 'speaker', 'relevancy']].copy()

        # Add the content compared to and the speaker of that content
        speaker_df['compared_to'] = scoreboard_df.shift(1).loc[speaker_mask, 'content']
        speaker_df['compared_to_speaker'] = scoreboard_df.shift(1).loc[speaker_mask, 'speaker']

        # Sort the dataframe by relevancy
        speaker_df.sort_values(by='relevancy', ascending=False, inplace=True)

        # Add the dataframe to the list
        speaker_dfs.append(speaker_df)

    return speaker_dfs[0], speaker_dfs[1], speaker_dfs[2]

In [ ]:
df_speaker1, df_speaker2, df_speaker3 = generate_speaker_dfs(scoreboard_df)

display(df_speaker1)
display(df_speaker2)
display(df_speaker3)

In [ ]:
all_spakers_by_relevancy = generate_speaker_dfs(scoreboard_df)



In [ ]:
type(all_spakers_by_relevancy)

In [ ]:
from sklearn.decomposition import LatentDirichletAllocation
from sklearn.feature_extraction.text import CountVectorizer
from nltk.stem import SnowballStemmer
from nltk.corpus import stopwords
from spacy.lang.en import English

# Loading the English tokenizer, tagger, parser, NER and word vectors
nlp = English()
# Create list of punctuation marks
punctuations = string.punctuation

# Create a list of stopwords
nlp.Defaults.stop_words |= {"'s", "'m", "ca"}
stop_words = nlp.Defaults.stop_words

# Load English tokenizer, tagger, parser, NER and word vectors
parser = English()

# Creating our tokenizer function
def spacy_tokenizer(sentence):
    # Creating our token object, which is used to create documents with linguistic annotations.
    mytokens = parser(sentence)

    # Lemmatizing each token and converting each token into lowercase
    mytokens = [word.lemma_.lower().strip() if word.lemma_ != "-PRON-" else word.lower_ for word in mytokens]

    # Removing stop words
    mytokens = [word for word in mytokens if word not in stop_words and word not in punctuations]

    # return preprocessed list of tokens
    return mytokens

# Function to apply LDA to all documents and return the model
def get_lda_model(data, n_topics=10, no_top_words=10, content_col='content'):
    vectorizer = CountVectorizer(tokenizer = spacy_tokenizer, ngram_range=(1,1))
    dtm = vectorizer.fit_transform(data[content_col])

    # Initialize LDA model
    lda = LatentDirichletAllocation(n_components=n_topics, random_state=42).fit(dtm)

    # Get the topic word distribution
    topic_word_dist = {}
    for i, topic in enumerate(lda.components_):
        topic_word_dist[i] = " ".join([vectorizer.get_feature_names()[j] for j in topic.argsort()[:-no_top_words - 1:-1]])

    return lda, vectorizer, topic_word_dist

lda_model, vectorizer, topic_word_dict = get_lda_model(scoreboard_df, n_topics=5, content_col='content')


In [ ]:
from top2vec import Top2Vec

In [ ]:
scoreboard_df['content'].apply(lambda x: len(x.split())).describe()


In [ ]:
# Change n_topics to a smaller number
scoreboard_df = apply_topic_modeling(scoreboard_df, n_topics=3)

scoreboard_df

In [ ]:
def get_word_count_points(row):
    word_count = len(nltk.word_tokenize(row['content']))
    return word_count

scoreboard_df['word_count_points'] = scoreboard_df.apply(get_word_count_points, axis=1)

#scoreboard_df

In [ ]:
def get_sentence_count_points(row):
    sentence_count = len(nltk.sent_tokenize(row['content']))
    return sentence_count

scoreboard_df['sentence_count_points'] = scoreboard_df.apply(get_sentence_count_points, axis=1)

#scoreboard_df

In [ ]:
def get_named_entity_points(row):
    named_entity_count = len(nlp(row['content']).ents)
    return named_entity_count

scoreboard_df['named_entity_count_points'] = scoreboard_df.apply(get_named_entity_points, axis=1)

#scoreboard_df

In [ ]:
def get_named_entities(row):
    named_entities = nlp(row['content']).ents
    return named_entities

scoreboard_df['named_entities'] = scoreboard_df.apply(get_named_entities, axis=1)

#scoreboard_df

In [ ]:
def get_word_bonus_points(row):
    sentence_count = len(set(word.lower() for word in nltk.word_tokenize(row['content']) if word.lower() in english_vocab))
    return sentence_count

scoreboard_df['bonus_word_points'] = scoreboard_df.apply(get_word_bonus_points, axis=1)

#scoreboard_df

In [ ]:
def get_bonus_word_words(row):
    sentence_count = (set(word.lower() for word in nltk.word_tokenize(row['content']) if word.lower() in english_vocab))
    return sentence_count

scoreboard_df['bonus_word_words'] = scoreboard_df.apply(get_bonus_word_words, axis=1)

#scoreboard_df

In [ ]:
def calc_base_score(row):
    base_score = row['word_count_points'] + row['sentence_count_points'] +row['named_entity_count_points'] + row['bonus_word_points']
    return base_score

scoreboard_df['base_score'] = scoreboard_df.apply(calc_base_score, axis=1)

#scoreboard_df

In [ ]:
def prompt_epl(row):
    if not row['content']: # Ensure that there is a current sentence
        return "Error: A current sentence is required."

    # Initialize content with the current sentence
    content = f"Evaluate the following snippet for Ethos (E), Pathos (P), and Logos (L). (Scale: 0.0 - 9.9) For non-zero score only: provide a brief explanation, otherwise leave empty string. It should be equally difficult to score 0.0 as it would be to score 9.9. [Snippet: \'{row['content']}\'] Return only a JSON object in this schema: {{E: x.x, P: y.y, L: z.z, E_Expl: string, P_Expl:, string L_Expl: string}}. Explanations must be 1 sentence long."

    prompt = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {
                "role": "system",
                "content": "You are a master debater and assistant to the judge of the US Presidential Debates."
            },
            {
                "role": "user",
                "content": content
            }
        ]
    )
    return prompt

In [ ]:
def calc_epl_multiplier(row):
    epl_multiplier =  1 + ((row['E'] + row['P'] + row['L']) / 30)
    return epl_multiplier

scoreboard_df['epl_multiplier'] = scoreboard_df.apply(calc_epl_multiplier, axis=1)

#scoreboard_df

In [ ]:
all_data = pd.read_csv("sep-29-2008-final-scoreboard-csv-2.csv")

In [4]:
from nltk.tokenize import sent_tokenize, word_tokenize
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer


In [ ]:
tokens = nltk.word_tokenize(text.lower())  # tokenize and convert to lowercase

def lexical_diversity(text):
    types = Counter(tokens)  # count unique tokens
    return len(types) / len(tokens)

text = "Jabari is a person person"
print(lexical_diversity(text))

In [ ]:
def avg_sentence_length(text):
    sentences = sent_tokenize(text)
    return sum(len(word_tokenize(sentence)) for sentence in sentences) / len(sentences)

print(avg_sentence_length(text))

In [1]:
scoring_rules = {
    'ADJ': {'ethos': 0, 'pathos': 2, 'logos': 1, 'weight': 1, 'decay': 0.75},  # adjectives
    'ADV': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 0.5, 'decay': 0.75},  # adverbs
    'AUX': {'ethos': 1, 'pathos': 0, 'logos': 1, 'weight': 0.25, 'decay': 0.75},  # auxiliary
    'CCONJ': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.75, 'decay': 0.75},  # coordinating conjunction
    'DET': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.25, 'decay': 0.75},  # determiner
    'INTJ': {'ethos': 0, 'pathos': 2, 'logos': 0, 'weight': 1, 'decay': 0.75},  # interjection
    'NOUN': {'ethos': 1, 'pathos': 0, 'logos': 0, 'weight': 1, 'decay': 0.75},  # noun
    'NUM': {'ethos': 1, 'pathos': 1, 'logos': 2, 'weight': 0.5, 'decay': 0.75},  # numeral
    'PART': {'ethos': 0, 'pathos': 0, 'logos': 1, 'weight': 0.1, 'decay': 0.75},  # particle
    'PRON': {'ethos': 2, 'pathos': 1, 'logos': 1, 'weight': 1, 'decay': 0.75},  # pronoun
    'PROPN': {'ethos': 2, 'pathos': 1, 'logos': 0, 'weight': 0.5, 'decay': 0.75},  # proper noun
    'PUNCT': {'ethos': 0, 'pathos': 1, 'logos': 2, 'weight': 0.2, 'decay': 0.75},  # punctuation
    'SCONJ': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.5, 'decay': 0.75},  # subordinating conjunction
    'SYM': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 0.1, 'decay': 0.75},  # symbol
    'VERB': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 1, 'decay': 0.75},  # verb
    'X': {'ethos': 0, 'pathos': 2, 'logos': 0, 'weight': 0.1, 'decay': 0.75},  # other
}

In [2]:
def score_text(text):
    # Initialize the scores
    scores = defaultdict(int)

    # Process the text
    doc = nlp(text)

    # Prepare a dictionary to hold temporary scores
    temp_scores = {'ethos': 0, 'pathos': 0, 'logos': 0}

    # Score each token
    for token in doc:
        pos = token.pos_
        if pos in scoring_rules:
            # Apply the ethos, pathos, logos scores, weight and decay
            temp_scores['ethos'] += scoring_rules[pos]['ethos'] * scoring_rules[pos]['weight']
            temp_scores['pathos'] += scoring_rules[pos]['pathos'] * scoring_rules[pos]['weight']
            temp_scores['logos'] += scoring_rules[pos]['logos'] * scoring_rules[pos]['weight']

            # Apply the decay
            temp_scores['ethos'] *= scoring_rules[pos]['decay']
            temp_scores['pathos'] *= scoring_rules[pos]['decay']
            temp_scores['logos'] *= scoring_rules[pos]['decay']

            # Add the temporary scores to the total scores
            scores['ethos'] += temp_scores['ethos']
            scores['pathos'] += temp_scores['pathos']
            scores['logos'] += temp_scores['logos']

    return scores


In [5]:
text = "Hi my name is beautiful Mr. Mr"

score_text(text)

NameError: name 'nlp' is not defined

In [ ]:

import matplotlib.pyplot as plt
import seaborn as sns


# Updated POS to EPL mapping
scoring_rules = {
    'ADJ': {'ethos': 0, 'pathos': 2, 'logos': 1, 'weight': 1, 'decay': 0.75},  # adjectives
    'ADV': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 0.5, 'decay': 0.75},  # adverbs
    'AUX': {'ethos': 1, 'pathos': 0, 'logos': 1, 'weight': 0.25, 'decay': 0.75},  # auxiliary
    'CCONJ': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.75, 'decay': 0.75},  # coordinating conjunction
    'DET': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.25, 'decay': 0.75},  # determiner
    'INTJ': {'ethos': 0, 'pathos': 2, 'logos': 0, 'weight': 1, 'decay': 0.75},  # interjection
    'NOUN': {'ethos': 1, 'pathos': 0, 'logos': 0, 'weight': 1, 'decay': 0.75},  # noun
    'NUM': {'ethos': 1, 'pathos': 1, 'logos': 2, 'weight': 0.5, 'decay': 0.75},  # numeral
    'PART': {'ethos': 0, 'pathos': 0, 'logos': 1, 'weight': 0.1, 'decay': 0.75},  # particle
    'PRON': {'ethos': 2, 'pathos': 1, 'logos': 1, 'weight': 1, 'decay': 0.75},  # pronoun
    'PROPN': {'ethos': 2, 'pathos': 1, 'logos': 0, 'weight': 0.5, 'decay': 0.75},  # proper noun
    'PUNCT': {'ethos': 0, 'pathos': 1, 'logos': 2, 'weight': 0.2, 'decay': 0.75},  # punctuation
    'SCONJ': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.5, 'decay': 0.75},  # subordinating conjunction
    'SYM': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 0.1, 'decay': 0.75},  # symbol
    'VERB': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 1, 'decay': 0.75},  # verb
    'X': {'ethos': 0, 'pathos': 2, 'logos': 0, 'weight': 0.1, 'decay': 0.75},  # other
}

def pos_frequency(text):
    doc = nlp(text)
    pos_frequency = Counter(token.pos_ for token in doc)

    result = defaultdict(list)
    epl_scores = defaultdict(int)

    for pos, freq in pos_frequency.items():
        epl_score_tuples = scoring_rules.get(pos, [('none', 0)])
        for epl, score in epl_score_tuples:
            result[pos].append((epl, freq))
            epl_scores[epl] += score * freq

    return result, dict(epl_scores)

def pos_to_df(pos_freq, epl_scores):
    data = [(pos, epl, freq) for pos, epl_freq in pos_freq.items() for epl, freq in epl_freq]
    df_pos = pd.DataFrame(data, columns=['POS', 'EPL', 'Frequency'])

    data = [(epl, score) for epl, score in epl_scores.items()]
    df_scores = pd.DataFrame(data, columns=['EPL', 'Score'])

    return df_pos, df_scores

def visualize(df_pos, df_scores):
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_pos, x='POS', y='Frequency', hue='EPL')
    plt.title('POS Frequency by EPL')
    plt.show()

    plt.figure(figsize=(10, 6))
    sns.barplot(data=df_scores, x='EPL', y='Score')
    plt.title('EPL Scores')
    plt.show()

# Example usage
text = "I really enjoy using artificial intelligence. It's amazing how it can understand language and generate text."

pos_freq, epl_scores = pos_frequency(text)
df_pos, df_scores = pos_to_df(pos_freq, epl_scores)
visualize(df_pos, df_scores)


In [ ]:


# Tokenize text into sentences
def tokenize_into_sentences(text):
    doc = nlp(text)
    return [sent.text for sent in doc.sents]

# Create a TF-IDF vectorizer
vectorizer = TfidfVectorizer()

def calculate_tfidf_scores(text):
    # Tokenize the text into sentences
    sentences = tokenize_into_sentences(text)

    # Calculate TF-IDF values for each word in the text
    tfidf_matrix = vectorizer.fit_transform(sentences)
    tfidf_scores = dict(zip(vectorizer.get_feature_names_out(), tfidf_matrix.sum(axis=0).A1))
    
    # Initialize EPL scores
    epl_scores = {'ethos': 0, 'pathos': 0, 'logos': 0}

    # Assign EPL scores based on POS tags of words with high TF-IDF scores
    for word, score in tfidf_scores.items():
        if score > 0.2:  # TF-IDF threshold
            token = nlp(word)[0]
            if token.pos_ in ['NOUN', 'PRON', 'PROPN']:
                epl_scores['ethos'] += 1
            elif token.pos_ in ['VERB', 'ADJ', 'ADV']:
                epl_scores['pathos'] += 1
            elif token.pos_ in ['CONJ', 'DET', 'AUX']:
                epl_scores['logos'] += 1

    return epl_scores

text = "Without further evidence, I would have zero chance of winning."

epl_scores = calculate_tfidf_scores(text)

print(f"EPL scores from TF-IDF: {epl_scores}")

In [ ]:
# POS to EPL mapping
scoring_rules = {
    'ADJ': {'ethos': 0, 'pathos': 2, 'logos': 1, 'weight': 1, 'decay': 0.75},  # adjectives
    'ADV': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 0.5, 'decay': 0.75},  # adverbs
    'AUX': {'ethos': 1, 'pathos': 0, 'logos': 1, 'weight': 0.25, 'decay': 0.75},  # auxiliary
    'CCONJ': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.75, 'decay': 0.75},  # coordinating conjunction
    'DET': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.25, 'decay': 0.75},  # determiner
    'INTJ': {'ethos': 0, 'pathos': 2, 'logos': 0, 'weight': 1, 'decay': 0.75},  # interjection
    'NOUN': {'ethos': 1, 'pathos': 0, 'logos': 0, 'weight': 1, 'decay': 0.75},  # noun
    'NUM': {'ethos': 1, 'pathos': 1, 'logos': 2, 'weight': 0.5, 'decay': 0.75},  # numeral
    'PART': {'ethos': 0, 'pathos': 0, 'logos': 1, 'weight': 0.1, 'decay': 0.75},  # particle
    'PRON': {'ethos': 2, 'pathos': 1, 'logos': 1, 'weight': 1, 'decay': 0.75},  # pronoun
    'PROPN': {'ethos': 2, 'pathos': 1, 'logos': 0, 'weight': 0.5, 'decay': 0.75},  # proper noun
    'PUNCT': {'ethos': 0, 'pathos': 1, 'logos': 2, 'weight': 0.2, 'decay': 0.75},  # punctuation
    'SCONJ': {'ethos': 0, 'pathos': 0, 'logos': 2, 'weight': 0.5, 'decay': 0.75},  # subordinating conjunction
    'SYM': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 0.1, 'decay': 0.75},  # symbol
    'VERB': {'ethos': 0, 'pathos': 1, 'logos': 1, 'weight': 1, 'decay': 0.75},  # verb
    'X': {'ethos': 0, 'pathos': 2, 'logos': 0, 'weight': 0.1, 'decay': 0.75},  # other
}

def calculate_epl_scores(text):
    # Tokenize the text into sentences
    sentences = tokenize_into_sentences(text)

    # Initialize EPL scores
    epl_scores = {'ethos': 0, 'pathos': 0, 'logos': 0}

    # Assign EPL scores based on POS tags of words with high TF-IDF scores
    for sentence in sentences:
        doc = nlp(sentence)
        pos_counts = defaultdict(int)
        for token in doc:
            pos_counts[token.pos_] += 1
            rule = scoring_rules.get(token.pos_, {})
            for epl, value in rule.items():
                if epl in ['ethos', 'pathos', 'logos']:
                    decayed_value = value * (rule['weight'] * rule['decay'] ** (pos_counts[token.pos_] - 1))
                    epl_scores[epl] += decayed_value

    return epl_scores


In [ ]:
calculate_epl_scores("CMV: There is a moral imperative to change and dismantle old cultures and traditions at odds with modern moral values")

In [ ]:
e_text_sample = "Why are moral rules needed?  For example, why do humans need rules about keeping promises, telling the truth and private property?  This answer should be fairly obvious.  Without such rules people would not be able to live amongst other humans.  People could not make plans, could not leave their belongings behind them wherever they went.  We would not know who to trust and what to expect from others.  Civilized, social life would not be possible.  So, the question is :   Why should humans care about being moral?"
p_text_sample = "If you grew up not understanding your emotions, you might wonder what to do with all these indefinable feelings that seem to get in the way of everything. Learn how to understand your emotions using the emotion wheel deeply."
l_text_sample = "Even when an essay has a clear thesis, and each paragraph in the body has an effective topic sentence, readers may have difficulty understanding and accepting what you have to say if the sentences do not flow smoothly. On the other hand, if your sentences are logically ordered and connected within each paragraph, your readers can easily follow what you are saying."

In [ ]:
e_sample_score = calculate_epl_scores(e_text_sample)
p_sample_score = calculate_epl_scores(p_text_sample)
l_sample_score = calculate_epl_scores(l_text_sample)

print(e_sample_score, p_sample_score, l_sample_score)

In [ ]:
core_words_epl = {
    "serenity": {"pathos": 1.75},
    "acceptance": {"ethos": 1.75},
    "apprehension": {"pathos": 1.75},
    "distraction": {"logos": 1.75},
    "pensiveness": {"pathos": 1.75},
    "boredom": {"ethos": 1.75},
    "annoyance": {"pathos": 1.75},
    "interest": {"logos": 1.75},
    "ecstacy": {"pathos": 1.25},
    "admiration": {"ethos": 1.25},
    "terror": {"pathos": 1.25},
    "amazement": {"logos": 1.25},
    "grief": {"pathos": 1.25},
    "loathing": {"ethos": 1.25},
    "rage": {"pathos": 1.25},
    "vigilance": {"logos": 1.25},
    "love": {"pathos": 1.75},
    "submission": {"ethos": 0.875, "pathos": 0.875},
    "awe": {"pathos": 0.875, "logos": 0.875},
    "disapproval": {"logos": 0.875, "pathos": 0.875},
    "remorse": {"pathos": 0.875, "ethos": 0.875},
    "contempt": {"ethos": 0.875, "pathos": 0.875},
    "aggressiveness": {"logos": 0.875, "pathos": 0.875},
    "optimism": {"pathos": 0.875, "logos": 0.875},
}


In [ ]:
import gensim.downloader
from gensim.models import KeyedVectors
from nltk.tokenize import word_tokenize
from scipy.spatial.distance import cosine

# Load pre-trained Word2Vec model (you need to download it first)
word_vectors = gensim.downloader.load('word2vec-google-news-300')

In [ ]:
type(word_vectors)

In [ ]:
def associate_emotion(text):
    # Tokenize the text
    words = word_tokenize(text)

    # Initialize EPL scores
    scores = {"ethos": 0, "pathos": 0, "logos": 0}

    # For each word in the text
    for word in words:
        # Skip if word not in Word2Vec vocab
        if word not in word_vectors.key_to_index:
            continue

        # Get Word2Vec vector for the word
        word_vector = word_vectors[word]

        # For each core word and its associated concepts
        for core_word, concepts in core_words_epl.items():
            # Skip if core word not in Word2Vec vocab
            if core_word not in word_vectors.key_to_index:
                continue

            # Get Word2Vec vector for the core word
            core_vector = word_vectors[core_word]

            # Calculate cosine similarity between word and core word
            similarity = 1 - cosine(word_vector, core_vector)
            #print(f'Word: {word}, Core Word: {core_word}, Similarity: {similarity}')  # Debug line

            # If similarity is above a threshold (e.g., 0.5)
            if similarity > 0.35:
                # Add weighted scores to EPL scores
                for concept, weight in concepts.items():
                    if concept in scores:
                        scores[concept] += weight * similarity

    return scores


In [ ]:
import nltk
nltk.download('averaged_perceptron_tagger')

degree_2_words_epl = {}

# For each core word and its associated EPL values
for core_word, epl_values in core_words_epl.items():
    try:
        # Get top 10 most similar words
        similar_words = word_vectors.most_similar(core_word, topn=10)

        # For each similar word and its similarity score
        for similar_word, _ in similar_words:
            # If the similar word is not already a core word
            if similar_word not in core_words_epl:
                # Exclude words with underscores
                if "_" not in similar_word:
                    # Exclude proper nouns
                    pos_tag = nltk.pos_tag([similar_word])[0][1]
                    if pos_tag != 'NNP' and pos_tag != 'NNPS':  # These are tags for proper nouns
                        # Add the similar word to the 2nd degree word list, with 90% of the original score
                        degree_2_words_epl[similar_word] = {k: v*0.9 for k, v in epl_values.items()}  # Adjusting EPL values here
    except KeyError:
        # Ignore core words that are not in the Word2Vec vocabulary
        continue


In [ ]:
degree_2_words_epl

In [ ]:
degree_3_words_epl = {}

# For each 2nd degree word and its associated EPL values
for degree_2_word, epl_values in degree_2_words_epl.items():
    try:
        # Get top 20 most similar words
        similar_words = word_vectors.most_similar(degree_2_word, topn=20)

        # For each similar word and its similarity score
        for similar_word, _ in similar_words:
            # If the similar word is not already a 2nd degree or core word
            if similar_word not in degree_2_words_epl and similar_word not in core_words_epl:
                # Exclude words with underscores
                if "_" not in similar_word:
                    # Exclude proper nouns
                    pos_tag = nltk.pos_tag([similar_word])[0][1]
                    if pos_tag != 'NNP' and pos_tag != 'NNPS':  # These are tags for proper nouns
                        # Add the similar word to the 3rd degree word list, with 85% of the original score
                        degree_3_words_epl[similar_word] = {k: v*0.85 for k, v in epl_values.items()}  # Adjusting EPL values here
    except KeyError:
        # Ignore 2nd degree words that are not in the Word2Vec vocabulary
        continue


In [ ]:
e_text_sample = "Why are moral rules needed?  For example, why do humans need rules about keeping promises, telling the truth and private property?  This answer should be fairly obvious.  Without such rules people would not be able to live amongst other humans.  People could not make plans, could not leave their belongings behind them wherever they went.  We would not know who to trust and what to expect from others.  Civilized, social life would not be possible.  So, the question is :   Why should humans care about being moral?"
p_text_sample = "If you grew up not understanding your emotions, you might wonder what to do with all these indefinable feelings that seem to get in the way of everything. Learn how to understand your emotions using the emotion wheel deeply."
l_text_sample = "Even when an essay has a clear thesis, and each paragraph in the body has an effective topic sentence, readers may have difficulty understanding and accepting what you have to say if the sentences do not flow smoothly. On the other hand, if your sentences are logically ordered and connected within each paragraph, your readers can easily follow what you are saying."

In [ ]:
print(degree_2_words_epl)

In [ ]:
print(associate_emotion(p_text_sample), associate_emotion(e_text_sample), associate_emotion(l_text_sample))

In [ ]:

pd.set_option('display.max_colwidth', None)
